In [ ]:
%pip install brokenaxes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import os
import numpy as np
from brokenaxes import brokenaxes
%matplotlib inline

## Configuration

In [ ]:
dataset_name = "" # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
model_name = "CLIP" # CLIP
fine_tuned_accuracy = 0
base_accuracy = 0
linear_probe_accuracy = 0
best_aug_acc_num_images = 0
transformation = ["Standard"] # "Standard"

results_path = f"./Data/{dataset_name}/{domain}/Entire_Transformation_Matrix_W"
graphed_results_path = f"./Graphs/{dataset_name}/{domain}"
indices = [i for i in range(12)]

## File Prepping

In [ ]:
results = [] # File Loading

try:
    for filename in os.listdir(results_path):
        if filename in [".DS_Store", "Base_Fine_Tuned_Classifier_Results.json", "Base_Linear_Probe_Results.json"]:
            continue
        file_path = os.path.join(results_path, filename)
        if os.path.isfile(file_path):
            results.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

data = [pd.read_json(i) for i in results]
data = sorted(data, key=lambda df: df["Train_Data_Size"][0])

In [ ]:
num_data_files = len(results) 

## Classification Acccuracy and Cosine Similarity Graphs

In [ ]:
# figure, axes = plt.subplots(nrows, ncols)
for i in indices: # For Each Layer
    plt.figure(figsize=(6,4))
    train_size = []
    acc = []
    co_sim_cls = []
    for j in range(num_data_files): # For Each File
        train_size.append(data[j]["Train_Data_Size"][i][0])
        acc.append(data[j]["Classification_Accuracy"][i])
        co_sim_cls.append(data[j]["CLS_Cosine_Similarity"][i])
    
    plt.plot(train_size, acc, marker='o', color='steelblue', label="Accuracy")
    plt.plot(train_size, co_sim_cls, marker='o', color="forestgreen", label="Cosine Similarity")
    plt.legend()
    plt.title(f"Augmented {model_name} - {dataset_name}: {domain} with W at Layer {i}")
    plt.xlabel("Size of Training Data", labelpad=6)
    plt.tight_layout()
    plt.savefig(f'./{graphed_results_path}/Augmented_Layer_{i}', dpi=600)
    plt.show()

In [ ]:
# All Accuracy Together
train_size = []
for i in range(num_data_files):
    train_size.append(data[i]["Train_Data_Size"][0][0])
acc_vals = {}
for i in indices:
    acc_vals[i] = []
for i in indices:
    for j in range(num_data_files):
        acc_vals[i].append(data[j]["Classification_Accuracy"][i])

colors = plt.cm.tab20(np.linspace(0,1,len(indices)))
plt.figure(figsize=(20,12))
for i in indices:
    plt.plot(train_size, acc_vals[i], marker="o", color=colors[i], label=f"{i} Layer")
length = train_size[-1]
plt.plot([0, length], [fine_tuned_accuracy, fine_tuned_accuracy], label="Fine-Tuned")
plt.plot([0, length], [base_accuracy, base_accuracy], label="Base")
plt.plot([0, length], [linear_probe_accuracy, linear_probe_accuracy], label="Linear Probe")

plt.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
plt.title(f"Augmented {model_name} - {dataset_name}: {domain} with W at All Layers - Classification Accuracy")
plt.xlabel("Size of training Data", labelpad=6)
plt.ylabel("Classification Accuracy")
plt.tight_layout()
plt.savefig(f'./{graphed_results_path}/All_Augmented_Layers_Classifiction_Accuracy', dpi=600)
plt.show()

In [ ]:
cls_vals = {}
for i in indices:
    cls_vals[i] = []
for i in indices:
    for j in range(num_data_files):
        cls_vals[i].append(data[j]["CLS_Cosine_Similarity"][i])

plt.figure(figsize=(20,12))
for i in indices:
    plt.plot(train_size, acc_vals[i], marker="o", color=colors[i], label=f"{i} Layer")
plt.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
plt.title(f"Augmented {model_name} - {dataset_name}: {domain} with W at All Layers - CLS Cosine Similarity")
plt.xlabel("Size of training Data", labelpad=6)
plt.ylabel("Cosine Similarity")
plt.tight_layout()
plt.savefig(f'./{graphed_results_path}/All_Augmented_Layers_CLS_Cosine_Similarity', dpi=600)
plt.show()

## Ablation Graphs

In [ ]:
results = []

try:
    for filename in os.listdir(results_path):
        if filename not in [f"{transformation[0]}_{best_aug_acc_num_images}_Results.json"]:
            continue
        file_path = os.path.join(results_path, filename)
        if os.path.isfile(file_path):
            results.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

print(f"Number of Files: {len(results)}")

results = [pd.read_json(i) for i in results]

In [ ]:
acc_aug = []
acc = []
linear_probe = []

for i in indices:
    acc_aug.append(results[0]["Classification_Accuracy"][i])
    acc.append(results[1]["Classification_Accuracy"][i])
    linear_probe.append(results[2]["Classification_Accuracy"][i])

In [ ]:
for i in linear_probe:
    print(i)

In [ ]:
plt.figure(figsize=(10,4))

plt.bar([i for i in indices], fine_tuned_accuracy, color="dodgerblue", label="Fine-Tuned")

plt.bar([i for i in indices], acc, color="teal", label=f"{best_aug_acc_num_images} Images Augmented Base")
plt.bar([i for i in indices], linear_probe, color="cadetblue", label="Linear Probe")
plt.bar([i for i in indices], acc_aug, color="lime", label="Base with Full-Fined Tuned Model's Classification Head")



plt.bar([i for i in indices], base_accuracy, color="cyan", label="Base Accuracy")

plt.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
plt.title(f"{model_name} - {dataset_name}: {domain} Results")
plt.xlabel("Layer")
plt.ylabel("Classification Accuracy")
plt.tight_layout()
plt.savefig(f'./{graphed_results_path}/Ablation_Classification_Accuracy', dpi=600)
plt.show()

## Residual Graphs

In [ ]:
# Structured in a way like so: data[file_num]["Residuals"][0][0], where the first 0 is just because there are accidental copies and the second 0 for the indice of 0-11
# shape (12) -> (12) -> 768

In [ ]:
print(type(data[30]["Residuals"][0][5][0]))

In [ ]:
print(type(data[30]["Residuals"][0][10][0]))

In [ ]:
residuals = {i: [] for i in indices}

for i in indices:
    for j in range(num_data_files):
        residuals[i].append(data[j]["Residuals"][0][i]) # Adds the list of numbers

In [ ]:
print(len(residuals))

In [ ]:
print(len(residuals[0][-1]))

In [ ]:
# Instead of each being a vector of 768 dimensions, I reduce it to a single scalar for MSE 

In [ ]:
residuals_mse = {}
for i in indices:
    residuals_mse[i] = []
    for j in range(num_data_files):
        residuals_mse[i].append(sum(residuals[i][j]) / (num_data_files * len(residuals[0][-1])))


In [ ]:
plt.figure(figsize=(20,12))

for i in indices:
    plt.plot(train_size, residuals_mse[i], marker="o", label=f"{i} Layer MSE Residuals")
plt.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
plt.title(f"{model_name} - {dataset_name}: {domain} with W at All Layers - Residuals MSE")
plt.xlabel("Size of training Data", labelpad=6)
plt.tight_layout()
plt.savefig(f'./{graphed_results_path}/All_Augmented_Layers_Residuals', dpi=600)
plt.show()